# Module 3.1 — How Embeddings Work

An **embedding** maps text to a dense vector in ℝⁿ such that semantically similar texts are geometrically close.

## Distance Metrics
| Metric | Formula | Best For |
|---|---|---|
| Cosine similarity | cos(θ) = a·b / (|a||b|) | Most common in NLP |
| Euclidean | √Σ(aᵢ−bᵢ)² | Magnitude matters |
| Dot product | Σ aᵢbᵢ | Fast; works for normalised vectors |

In [ ]:
import numpy as np
from openai import OpenAI

client = OpenAI()

def get_embedding(text: str, model: str = "text-embedding-3-small") -> np.ndarray:
    resp = client.embeddings.create(input=text, model=model)
    return np.array(resp.data[0].embedding)

def cosine_similarity(a: np.ndarray, b: np.ndarray) -> float:
    return float(np.dot(a, b) / (np.linalg.norm(a) * np.linalg.norm(b)))

def euclidean_distance(a: np.ndarray, b: np.ndarray) -> float:
    return float(np.linalg.norm(a - b))

def dot_product(a: np.ndarray, b: np.ndarray) -> float:
    return float(np.dot(a, b))

# ── Embed a set of sentences ──────────────────────────────────────────────────
sentences = [
    "The cat sat on the mat.",
    "A feline rested on a rug.",          # semantically similar
    "Machine learning transforms data.",   # unrelated
    "Deep learning is a subset of ML.",   # somewhat related to prev
]

embeds = [get_embedding(s) for s in sentences]
print(f"Embedding dimension: {len(embeds[0])}\n")

# ── Pairwise cosine similarity ────────────────────────────────────────────────
print("Cosine Similarity Matrix:")
print(f"{'':45}", end="")
for s in sentences:
    print(f"{s[:20]:>22}", end="")
print()

for i, (s1, e1) in enumerate(zip(sentences, embeds)):
    print(f"{s1[:45]:45}", end="")
    for j, (s2, e2) in enumerate(zip(sentences, embeds)):
        sim = cosine_similarity(e1, e2)
        print(f"{sim:>22.4f}", end="")
    print()


In [ ]:
# ── Visualise in 2D using PCA ─────────────────────────────────────────────────
# !pip install matplotlib scikit-learn
import matplotlib.pyplot as plt
from sklearn.decomposition import PCA

pca  = PCA(n_components=2)
pts  = pca.fit_transform(np.array(embeds))

plt.figure(figsize=(8, 5))
for (x, y), label in zip(pts, sentences):
    plt.scatter(x, y, s=100)
    plt.annotate(label[:35], (x, y), textcoords="offset points", xytext=(6, 4), fontsize=8)

plt.title("Embeddings projected to 2D (PCA)")
plt.xlabel("PC1"); plt.ylabel("PC2")
plt.tight_layout(); plt.savefig("embeddings_pca.png", dpi=120); plt.show()
print("Plot saved to embeddings_pca.png")
